In [13]:
# 模型初始化
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="MiniMax-M3",
    api_key='sk-cp-GsTRU_GlFRX_WPY8cf8IqBTDtrVJP4zl4EWLZ8D7627lrDtV-MbWYnWWK7rmvT7ZRluuMtUL7VP2sCD2uq6SzkpIlY0EAJtCIHW2uSGAT4QKYxX9mJQmK8I',
    base_url="https://api.minimaxi.com/v1",
)

In [14]:
# 定义相关工具
from datetime import timedelta
from langchain_core.tools import tool
@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气情况
    支持中国主要城市的天气查询
    
    Args:
        city (str): 城市名称
        
    Returns:
        str: 城市的天气情况
    
    Example:
        >>> get_weather("北京")
        "多云，15-22°C，空气质量良"
        
    """
    weather_db = {
        "北京": "多云，15-22°C，空气质量良，湿度 45%，风力 3-4 级",
        "上海": "晴，18-25°C，空气质量良，湿度 50%，风力 2-3 级",
        "广州": "阵雨，22-28°C，空气质量良，湿度 60%，风力 3-4 级",
        "深圳": "多云，20-26°C，空气质量良，湿度 55%，风力 2-3 级",
        "成都": "阴，16-24°C，空气质量良，湿度 70%，风力 2-3 级",
        "杭州": "晴，17-23°C，空气质量良，湿度 50%，风力 2-3 级",
        "重庆": "多云，19-27°C，空气质量良，湿度 65%，风力 3-4 级",
        "武汉": "阵雨，21-29°C，空气质量良，湿度 60%，风力 2-3 级",
    }
    result = weather_db.get(city)
    if result:
        return f"{city}:{result}"
    else:
        return f"抱歉，暂不支持查询{city}的天气情况。当前支持的城市有：北京、上海、广州、深圳、成都、杭州、重庆、武汉。"

@tool
def calculate(expression: str) -> str:
    """
    计算数学表达式的结果
    
    Args:
        expression (str): 数学表达式
        
    Returns:
        str: 表达式的计算结果或错误信息
    
    Example:
        >>> calculate("2 + 2")
        "4"
        
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"计算错误: {e}"

@tool
def get_time_info(query_type: str = "current") -> str:
    """
    获取当前时间或日期信息
    
    Args:
        query_type (str): 查询类型
            - "current": 获取当前时间
            - "date": 获取当前日期
            - "tomorrow": 获取明天的日期
            - "yesterday": 获取昨天的日期
            - "weekday": 获取今天是星期几
        
    Returns:
        str: 当前时间或日期信息
    
    Example:
        >>> get_time_info("current")
        "当前时间是 2026年1月1日 14:30:15"
        
        >>> get_time_info("date")
        "今天是 2024-06-15"
    """
    from datetime import datetime
    now = datetime.now()
    
    if query_type == "current":
        return f"当前时间是 {now.strftime('当前时间%Y年%m月%d日 %H:%M:%S')}"
    elif query_type == "date":
        return f"今天是 {now.strftime('%Y-%m-%d')}"
    elif query_type == "tomorrow":
        tomorrow = now + timedelta(days=1)
        return f"明天是 {tomorrow.strftime('%Y-%m-%d')}"
    elif query_type == "yesterday":
        yesterday = now - timedelta(days=1)
        return f"昨天是 {yesterday.strftime('%Y-%m-%d')}"
    elif query_type == "weekday":
        weekdays = ["星期一", "星期二", "星期三", "星期四", "星期五", "星期六", "星期日"]
        return f"今天是 {weekdays[now.weekday()]}"
    else:
        return "无效的查询类型，请使用 'current', 'date', 'tomorrow', 'yesterday', 或 'weekday'。"
    
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """
    支持主要货币之间之间的实时汇率转换
    
    Args:
        amount (float): 金额
        from_currency (str): 源货币代码（CYN/USD/EUR/GBP/JPY/HKD）
        to_currency (str): 目标货币代码CYN/USD/EUR/GBP/JPY/HKD）
        
    Returns:
        str: 转换后的金额或错误信息
    
    Example:
        >>> convert_currency(100, "USD", "CNY")
        "100 USD = 700 CNY"
        
    """
    # 模拟汇率数据(相对CNY)
    exchange_rates = {
        "CNY": 1.0,
        "USD": 0.14,
        "EUR": 0.12,
        "GBP": 0.10,
        "JPY": 15.0,
        "HKD": 1.09,
    }
    
    currency_names = {
        "CNY": "人民币",
        "USD": "美元",
        "EUR": "欧元",
        "GBP": "英镑",
        "JPY": "日元",
        "HKD": "港币",
    }

    from_currency = from_currency.upper()
    to_currency = to_currency.upper()

    if from_currency not in exchange_rates:
        return f"不支持的源货币转换: {from_currency}，当前支持的货币有：CNY, USD, EUR, GBP, JPY, HKD。"
    if to_currency not in exchange_rates:
        return f"不支持的目标货币转换: {to_currency}，当前支持的货币有：CNY, USD, EUR, GBP, JPY, HKD。"

    rate = exchange_rates[to_currency] / exchange_rates[from_currency]
    converted_amount = amount * rate
    from_currency_name = currency_names[from_currency]
    to_currency_name = currency_names[to_currency]
    return f"{amount} {from_currency_name} {from_currency} = {converted_amount:.2f} {to_currency_name} {to_currency}"

@tool
def search_info(keyword: str, category: str = "all") -> str:
    """
    支持主要搜索引擎的搜索功能
    
    Args:
        keyword (str): 搜索关键词
        category (str): 搜索类别
            - "all": 全部
            - "news": 新闻
            - "images": 图片
            - "videos": 视频
            - "products": 产品
        
    Returns:
        str: 搜索结果或错误信息
    
    Example:
        >>> search_info("Python", "news")
        "搜索结果: ..."
        
    """
    
    # 模拟数据库
    product_db = {
        "手机": "iPhone 15（¥5999），小米 13（¥3999），华为 P60（¥4999）",
        "笔记本": "MacBook Air（¥8999），ThinkPad X1 Carbon（¥10999），Dell XPS 13（¥9999）",
        "耳机": "AirPods Pro（¥1999），Sony WH-1000XM4（¥2499），Bose QuietComfort 35 II（¥2299）",
        "相机": "Canon EOS R6（¥14999），Sony A7 III（¥12999），Nikon Z6 II（¥13999）",
        "手表": "Apple Watch Series 9（¥3999），华为 Watch GT 3（¥1999），三星 Galaxy Watch 5（¥2499）",
    }
    
    news = {
        "AI": "最新AI技术突破，人工智能正在改变各行各业。",
        "经济": "全球经济复苏势头良好，股市持续上涨。",
        "科技": "科技公司发布新款智能手机，引发市场热议。",
        "体育": "奥运会圆满落幕，各国运动员表现出色。",
        "娱乐": "电影《未来战士》票房破亿，成为年度热门大片。",
    }
    
    result =[]
    
    if category in ["all", "products"]:
        for key, value in product_db.items():
            if keyword in key:
                result.append(f"【产品】{key}: {value}")
    
    if category in ["all", "news"]:
        for key, value in news.items():
            if keyword in key or keyword in value:
                result.append(f"【新闻】{key}: {value}")
    
    if result:
        return "\n".join(result)
    else:
        return f"未找到与关键词 '{keyword}' 相关的 {category} 结果。"

In [19]:
# 创建Agent  model、tools、system_prompt、ToolStrategy、invoke、---消息列表
from langchain.agents import create_agent
from rich import print as rprint
from langchain_tavily import TavilySearch

class SmartAssistant:
    """多功能智能助手"""
    def __init__(self):
        # 初始化模型
        self.model = model
        
        # 工具列表
        self.tools = [get_weather, calculate, get_time_info, convert_currency, search_info]
        
        # 系统提示词
        system_prompt = """你是一个多功能智能助手，能够提供:
        天气查询: 使用 get_weather 工具查询指定城市的天气情况。
        数学计算: 使用 calculate 工具计算数学表达式的结果。
        时间信息: 使用 get_time_info 工具获取当前时间、日期、明天、昨天或星期几。
        货币转换: 使用 convert_currency 工具进行主要货币之间的实时汇率转换。
        搜索功能: 使用 search_info 工具在主要搜索引擎中搜索信息。
        
        重要提示：
        1、仔细阅读用户的需求，确定需要使用哪个工具。
        2、如果需要调用多个工具，按顺序调用
        3、总是用友好、专业的语气回答
        4、如果工具返回了数据，要用通俗易懂的语言解释给用户听
        5、如果无法完成用户的请求，要诚实告诉用户，并提供可行的替代方案
        
        始终用中文回答用户的问题，除非用户明确要求使用其他语言。
        """
        
        self.agent = create_agent(
            model = self.model,
            tools = self.tools,
            system_prompt = system_prompt
        )
        
        self.messages = []
        
    def chat(self, user_input: str) -> str:
        """与用户进行对话"""
        self.messages.append({"role": "user", "content": user_input})
        result = self.agent.invoke({"messages": self.messages})
        self.messages = result['messages'] 
        
        # 返回最后一条AI消息
        for message in reversed(self.messages):
            if message.type == 'ai' and message.content:
                return message.content
        
        return "抱歉，我无法生成有效的回复。"
    
    def reset(self):
        """重置对话"""
        self.messages = []


In [23]:
# 主程序
def main():
    assistant = SmartAssistant()
    
    print("="*40)
    print("多功能智能助手（langchain1.2）")
    print("="*40)
    print("\n我可以帮你完成以下任务：")
    print("1. 查询天气")
    print("2. 进行数学计算")
    print("3. 获取时间信息")
    print("4. 进行货币转换")
    print("5. 搜索信息")
    print("输入 'exit' 退出对话，输入 'reset' 重置对话。")
    
    # demos = [
    #     "北京今天天气怎么样？",
    #     "请帮我计算 123 * 456",
    #     "现在几点了？",
    #     "请帮我把 100 美元转换成人民币",
    #     "帮我搜索一下关于人工智能的最新新闻",
    # ]
    
    # for demo in demos:
    #     print(f"\n[用户提问] {demo}")
    #     response = assistant.chat(demo)
    #     print(f"[Agent回复：] {response}")
        
    # 重置对话
    assistant.reset()
    
    # 交互模式
    print("="*40)
    print("进入交互模式")
    print("="*40)
    
    while True:
        user_input = input("\n[用户提问] :")
        if user_input.lower() == "exit":
            print("退出对话。")
            break
        elif user_input.lower() == "reset":
            assistant.reset()
            print("对话已重置。")
            continue
        
        response = assistant.chat(user_input)
        print(f"[Agent回复：] {response}")
        
    
if __name__ == "__main__":
    main()
    

多功能智能助手（langchain1.2）

我可以帮你完成以下任务：
1. 查询天气
2. 进行数学计算
3. 获取时间信息
4. 进行货币转换
5. 搜索信息
输入 'exit' 退出对话，输入 'reset' 重置对话。
进入交互模式
[Agent回复：] <think>终于找到一个新闻了。虽然不是非常具体的新闻，但至少能给用户一个回应。我应该诚实地告诉用户目前能搜索到的新闻很有限，并提供这个搜索到的内容。</think>

很抱歉，目前能搜索到的新闻比较有限。我为您找到了一条科技类新闻：

📰 **科技新闻**
科技公司发布新款智能手机，引发市场热议。

如果您想了解某个特定领域的新闻（比如国际新闻、体育新闻、财经新闻等），请告诉我具体的主题，我可以帮您进一步搜索。或者您也可以换个更具体的关键词再试试。

您对哪方面的新闻感兴趣呢？
[Agent回复：] <think>搜索iPhone相关的新闻都没有结果。我应该诚实告诉用户目前没有找到相关的新闻，并提供一些可行的替代方案。</think>

很抱歉，我尝试用多个关键词搜索（iPhone、苹果手机、Apple、iPhone 17），目前都没有找到相关的新闻结果。😅

**可能的原因：**
- 新闻数据可能没有及时更新
- iPhone近期没有重大新闻事件

**建议您可以：**
1. 稍后再试，我可以帮您重新搜索
2. 换一个搜索主题，比如搜索其他科技产品的新闻
3. 直接访问新闻网站（如新浪新闻、网易新闻、腾讯新闻等）查看iPhone的最新资讯

您想换个其他话题试试吗？
退出对话。
